# M16 · 01 — Tokenizer and pretraining data (A0)

Decision: `docs/decisions/0025-own-llm-m16.md` §4–§5. Commands: `TRAINING.md` §7.

1. What was downloaded (FineWeb2-HQ `spa_Latn`, FineWeb-Edu `sample/10BT`).
2. The tokenizer: byte-level BPE 32,768, Llama-3 split regex, trained on a ~2 GB 70/30 sample (`prepare_data sample` + `train-tokenizer`).
3. Acceptance (§4): fertility on held-out es/en text compared with Qwen3.5's tokenizer, lossless round-trip, accents/ñ, special tokens.
4. Projection: bytes/token → shard size and how many tokens the downloaded text gives.

Held-out text = the first 5,000 documents of each source's first raw file (`VAL_DOCS`). They are excluded from the tokenizer sample and become `val.bin`. Run from WSL with `experiments/.venv`, cwd = `experiments/m16-own-llm`.

In [1]:
import json, statistics, time
from pathlib import Path
import pyarrow.parquet as pq
from llm import HOME
from llm import prepare_data as P

for src in P.SOURCES:
    m = P._manifest(src)
    print(f"{src}: {len(m)} files, {sum(v['docs'] for v in m.values()):,} docs, "
          f"{sum(v['bytes'] for v in m.values()) / 1e9:.1f} GB of UTF-8 text (target {P.SOURCES[src]['target_bytes'] / 1e9:.0f} GB)")

es: 128 files, 11,549,695 docs, 37.8 GB of UTF-8 text (target 95 GB)
en: 14 files, 9,672,101 docs, 46.0 GB of UTF-8 text (target 60 GB)


In [2]:
def heldout(src, n=P.VAL_DOCS):
    """The first `n` docs of the source's first raw file -- or, once that raw file has been tokenized and
    deleted (cleanup, TRAINING.md §5b), the same docs decoded back from val.bin (the round-trip is lossless)."""
    first = sorted(P._manifest(src))[0]
    if (P.raw_dir(src) / first).exists():
        out = []
        for t in P._iter_texts(src, [first]):
            out.append(t)
            if len(out) == n:
                break
        return out
    import numpy as np
    from llm.tokenizer import EOT, load_tokenizer
    t = load_tokenizer(P.tokenizer_path())
    ids = np.fromfile(P.shard_dir(src) / 'val.bin', dtype=np.uint16)
    cuts = np.flatnonzero(ids == t.token_to_id(EOT))
    starts = np.concatenate([[0], cuts[:-1] + 1])
    return [t.decode(ids[s:e].tolist()) for s, e in zip(starts[:n], cuts[:n])]

held = {src: heldout(src) for src in P.SOURCES}
for src, docs in held.items():
    words = sum(len(d.split()) for d in docs)
    print(f"{src}: {len(docs)} held-out docs, {words:,} words, median {statistics.median(len(d.split()) for d in docs):.0f} words/doc")
print('\nexample es:', held['es'][3][:300].replace('\n', ' '))

es: 5000 held-out docs, 3,424,114 words, median 286 words/doc


en: 5000 held-out docs, 3,860,595 words, median 448 words/doc

example es:   Contenido Capítulo 1 Hechos & Prejuicios Capítulo 2 Variacion y selección natural frente a la evolucion Capítulo 3 Faltan los eslabones Capítulo 4 ¿Evolucionaron las aves? Capítulo 5 ¿Evolucionaron las ballenas? Capítulo 6 Humanos: ¿Imágenes de Dios o monos desarrollados? Capítulo 7 Astronomía Cap


In [3]:
from llm.tokenizer import SPECIALS, load_tokenizer
tok = load_tokenizer(P.tokenizer_path())
print('vocab', tok.get_vocab_size(), '| specials', {s: tok.token_to_id(s) for s in SPECIALS})

# Qwen3.5-9B's tokenizer (the app's current LLM) as the reference, vocab-only from its GGUF.
# The CUDA build of llama-cpp-python needs the backend's CUDA preload first (0014 §2).
import sys
BACKEND = Path('~/projects/new_youtube_suite/backend').expanduser()
sys.path.insert(0, str(BACKEND))
from app.llm import cuda_libs
cuda_libs.preload()
from llama_cpp import Llama
qwen_gguf = next((BACKEND / 'models/chapters/current/llm').glob('*.gguf'))
qwen = Llama(model_path=str(qwen_gguf), vocab_only=True, verbose=False)
print('reference:', qwen_gguf.name, '| vocab', qwen.n_vocab())

vocab 32768 | specials {'<|endoftext|>': 0, '<|im_start|>': 1, '<|im_end|>': 2, '<|pad|>': 3}


llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


reference: Qwen3.5-9B-Q4_K_M.gguf | vocab 248320


In [4]:
rows = []
for src, docs in held.items():
    words = sum(len(d.split()) for d in docs)
    nbytes = sum(len(d.encode()) for d in docs)
    ours = sum(len(e.ids) for e in tok.encode_batch(docs, add_special_tokens=False))
    ref = sum(len(qwen.tokenize(d.encode(), add_bos=False, special=False)) for d in docs)
    rows.append((src, words, nbytes, ours, ref))
    print(f"{src}: ours {ours / words:.3f} tok/word, {nbytes / ours:.2f} bytes/tok | "
          f"Qwen {ref / words:.3f} tok/word, {nbytes / ref:.2f} bytes/tok (vocab ~{qwen.n_vocab() // 1000}k) | "
          f"ours/Qwen = {ours / ref:.3f}")
BYTES_PER_TOKEN = {src: nbytes / ours for src, _, nbytes, ours, _ in rows}

es: ours 1.347 tok/word, 4.60 bytes/tok | Qwen 1.449 tok/word, 4.28 bytes/tok (vocab ~248k) | ours/Qwen = 0.930


en: ours 1.439 tok/word, 4.31 bytes/tok | Qwen 1.366 tok/word, 4.54 bytes/tok (vocab ~248k) | ours/Qwen = 1.054


In [5]:
bad = [d for src in held for d in held[src][:2000] if tok.decode(tok.encode(d).ids) != d]
print('round-trip failures on 4,000 held-out docs:', len(bad))

for s in [' canción', ' corazón', ' España', ' mañana', ' pingüino', ' acción', ' también', ' después',
          ' ¿Qué', ' ¡Hola', ' podcast', ' episodio', ' entrevista', ' YouTube', ' 1x08', ' 2026']:
    ids = tok.encode(s).ids
    print(f"{s!r:16} -> {len(ids)} token(s): {[tok.decode([i]) for i in ids]}")

round-trip failures on 4,000 held-out docs: 0
' canción'       -> 1 token(s): [' canción']
' corazón'       -> 1 token(s): [' corazón']
' España'        -> 1 token(s): [' España']
' mañana'        -> 1 token(s): [' mañana']
' pingüino'      -> 3 token(s): [' p', 'ingü', 'ino']
' acción'        -> 1 token(s): [' acción']
' también'       -> 1 token(s): [' también']
' después'       -> 1 token(s): [' después']
' ¿Qué'          -> 2 token(s): [' ¿', 'Qué']
' ¡Hola'         -> 2 token(s): [' ¡', 'Hola']
' podcast'       -> 2 token(s): [' pod', 'cast']
' episodio'      -> 1 token(s): [' episodio']
' entrevista'    -> 1 token(s): [' entrevista']
' YouTube'       -> 1 token(s): [' YouTube']
' 1x08'          -> 4 token(s): [' ', '1', 'x', '08']
' 2026'          -> 3 token(s): [' ', '202', '6']


In [6]:
total_tokens = 0
for src in P.SOURCES:
    text = sum(v['bytes'] for v in P._manifest(src).values())
    toks = text / BYTES_PER_TOKEN[src]
    total_tokens += toks
    print(f"{src}: {text / 1e9:.1f} GB text -> ~{toks / 1e9:.1f}B tokens -> ~{toks * 2 / 1e9:.0f} GB of uint16 shards")
print(f"total ~{total_tokens / 1e9:.1f}B tokens. Stage B needs 20-30B at 70/30: es {0.7 * 25:.1f}B / en {0.3 * 25:.1f}B for 25B")

es: 37.8 GB text -> ~8.2B tokens -> ~16 GB of uint16 shards
en: 46.0 GB text -> ~10.7B tokens -> ~21 GB of uint16 shards
total ~18.9B tokens. Stage B needs 20-30B at 70/30: es 17.5B / en 7.5B for 25B


In [7]:
# llama.cpp parity with the REAL 32k tokenizer (0025 §10): export a tiny random model with it and
# compare token ids on held-out documents. tests/test_export.py covers logits with a small tokenizer.
import tempfile, torch
from llm.config import ModelConfig
from llm.export_gguf import export
from llm.model import Decoder

cfg = ModelConfig(vocab_size=tok.get_vocab_size(), d_model=64, n_layers=1, n_heads=4, n_kv_heads=2, max_seq_len=128)
with tempfile.TemporaryDirectory() as d:
    gg = Path(d) / 'tiny32k.gguf'
    export(Decoder(cfg).state_dict(), cfg, P.tokenizer_path(), gg, f32=True)
    cpp = Llama(model_path=str(gg), vocab_only=True, verbose=False)
    n = mism = 0
    for src in held:
        for doc in held[src][:1000]:
            n += 1
            if cpp.tokenize(doc.encode(), add_bos=False, special=False) != tok.encode(doc).ids:
                mism += 1
print(f'llama.cpp vs ours, real tokenizer: {mism} mismatches in {n} held-out docs')

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


llama.cpp vs ours, real tokenizer: 0 mismatches in 2000 held-out docs


## Conclusions (2026-11-14)

**Tokenizer accepted (0025 §4).**
- **Training:** 32,768 tokens trained in 83 s on a 1.69 GB sample (peak 3.1 GB RAM). The sample
  was deleted afterwards.
- **Fertility on held-out text:**
  - **Spanish:** 1.347 tokens/word against Qwen3.5's 1.449, so our 32k vocab needs 7 % fewer
    tokens than Qwen's 248k. That is the 70 % share working.
  - **English:** 1.439 against 1.366, 5 % more. Acceptable for the 30 % share.
- **Round-trip:** lossless on 4,000 held-out documents.
- **Accents and ñ:** common words such as *canción, corazón, mañana, también, después, episodio,
  entrevista* are single tokens. Rare ones split cleanly (*pingüino* → ` p|ingü|ino`). Numbers are
  split into groups of up to 3 digits, as the Llama-3 regex intends.
- **Special tokens:** ids 0–3 (`<|endoftext|>`, `<|im_start|>`, `<|im_end|>`, `<|pad|>`).

**Projection:**
- Measured bytes per token: 4.60 (es) and 4.31 (en).
- The 95 GB Spanish target gives about 20.6B tokens and the 46 GB of English about 10.7B, about
  31B in total and about 62 GB of uint16 shards.
- That covers stage B up to 30B tokens at 70/30 (21B es / 9B en) without repeating data.
- The English sample-10BT is all the English there is in this download. At 25B it is used once,
  0.7 epochs.